In [5]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.power import TTestIndPower

# Load the dataset
file_path = '/Users/baselhussein/Projects/impossible_goals/agg/data/merged_approaches.csv'
df = pd.read_csv(file_path)

# Define function to create binary indicators for sequences
def create_sequence_indicators(df, sequences):
    for index, row in sequences.iterrows():
        sequence_label = f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}"
        df[sequence_label] = (
            (df['lvl1'] == row['lvl1']) & 
            (df['lvl2'] == row['lvl2']) & 
            (df['lvl3'] == row['lvl3'])
        ).astype(int)
    return df

# Remove outliers (3 SD from mean)
mean_levels_complete = df['len_levels_complete'].mean()
std_levels_complete = df['len_levels_complete'].std()
lower_threshold = mean_levels_complete - 3 * std_levels_complete
upper_threshold = mean_levels_complete + 3 * std_levels_complete

data_no_outliers = df[(df['len_levels_complete'] >= lower_threshold) & (df['len_levels_complete'] <= upper_threshold)]

# Extract unique sequences of behaviors
sequences_unique = df[['lvl1', 'lvl2', 'lvl3']].drop_duplicates().reset_index(drop=True)

# Create binary indicators for each unique sequence
data_no_outliers = create_sequence_indicators(data_no_outliers, sequences_unique)

# Prepare the predictors (X) and the response variable (y) without outliers
X_no_outliers = data_no_outliers[sequences_unique.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)]
y_no_outliers = data_no_outliers['len_levels_complete']

# Add a constant term to the model for the intercept
X_no_outliers_with_intercept = sm.add_constant(X_no_outliers)

# Fit the linear regression model using statsmodels without outliers
model_no_outliers = sm.OLS(y_no_outliers, X_no_outliers_with_intercept)
results_no_outliers = model_no_outliers.fit()

# Extract coefficients, p-values, and confidence intervals for the model without outliers
no_outliers_summary = results_no_outliers.summary2().tables[1]

# Calculate the number of observations (N) for each unique sequence without outliers
sequence_counts_no_outliers = data_no_outliers.groupby(['lvl1', 'lvl2', 'lvl3']).size().reset_index(name='N')
sequence_counts_no_outliers['Sequence'] = sequence_counts_no_outliers.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)
sequence_counts_no_outliers = sequence_counts_no_outliers[['Sequence', 'N']]

# Merge the sequence counts with the regression summary
sequence_labels = sequences_unique.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)
no_outliers_summary['Sequence'] = ['const'] + sequence_labels.tolist()
merged_summary = no_outliers_summary.reset_index().merge(sequence_counts_no_outliers, on='Sequence', how='left')

# Display the merged summary with coefficients and number of observations
merged_summary

/var/folders/t1/sdbl9_cd061431ynt1cns5hh0000gn/T/ipykernel_5158/3620668414.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[sequence_label] = (
/var/folders/t1/sdbl9_cd061431ynt1cns5hh0000gn/T/ipykernel_5158/3620668414.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[sequence_label] = (
/var/folders/t1/sdbl9_cd061431ynt1cns5hh0000gn/T/ipykernel_5158/3620668414.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index

,index,Coef.,Std.Err.,t,P>|t|,[0.025,0.975],Sequence,N
0,const,32.632284,1.827043,17.860705,3.949567e-26,28.980076,36.284492,const,NaN
1,low-returns_fast-reframe->low-returns_fast-ref...,7.948361,2.557523,3.107835,2.843757e-03,2.835945,13.060777,low-returns_fast-reframe->low-returns_fast-ref...,31.0
2,low-returns_fast-reframe->low-returns_slow-ref...,9.367716,7.278879,1.286972,2.028881e-01,-5.182555,23.917987,low-returns_fast-reframe->low-returns_slow-ref...,2.0
3,low-returns_fast-reframe->low-returns_slow-ref...,5.117716,5.306607,0.964405,3.385898e-01,-5.490041,15.725473,low-returns_fast-reframe->low-returns_slow-ref...,4.0
4,low-returns_fast-reframe->high-returns_fast-re...,-1.032284,4.816190,-0.214336,8.309882e-01,-10.659710,8.595142,low-returns_fast-reframe->high-returns_fast-re...,5.0
5,low-returns_slow-reframe->low-returns_fast-ref...,12.367716,10.130452,1.220845,2.267672e-01,-7.882769,32.618201,low-returns_slow-reframe->low-returns_fast-ref...,1.0
6,low-returns_fast-reframe->low-returns_slow-ref...,2.185898,3.516286,0.621650,5.364515e-01,-4.843058,9.214853,low-returns_fast-reframe->low-returns_slow-ref...,11.0
7,high-returns_fast-reframe->low-returns_fast-re...,2.867716,5.306607,0.540405,5.908540e-01,-7.740041,13.475473,high-returns_fast-reframe->low-returns_fast-re...,4.0
8,low-returns_slow-reframe->low-returns_slow-ref...,3.867716,7.278879,0.531362,5.970669e-01,-10.682555,18.417987,low-returns_slow-reframe->low-returns_slow-ref...,2.0
9,low-returns_fast-reframe->high-returns_fast-re...,7.367716,10.130452,0.727284,4.697894e-01,-12.882769,27.618201,low-returns_fast-reframe->high-returns_fast-re...,1.0


In [6]:
# Export the merged summary to a CSV file
output_file_path = '/Users/baselhussein/Downloads/regression.csv'  # Update with your desired output path
merged_summary.to_csv(output_file_path, index=False)

In [7]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Load the dataset
file_path = '/Users/baselhussein/Projects/impossible_goals/agg/data/merged_approaches.csv'
df = pd.read_csv(file_path)

# Remove outliers (3 SD from mean)
mean_levels_complete = df['len_levels_complete'].mean()
std_levels_complete = df['len_levels_complete'].std()
lower_threshold = mean_levels_complete - 3 * std_levels_complete
upper_threshold = mean_levels_complete + 3 * std_levels_complete

data_no_outliers = df[(df['len_levels_complete'] >= lower_threshold) & (df['len_levels_complete'] <= upper_threshold)]

# Extract unique sequences and create a sequence column
data_no_outliers['Sequence'] = data_no_outliers.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)

# Count the occurrences of each unique sequence
sequence_counts_no_outliers = data_no_outliers['Sequence'].value_counts().reset_index()
sequence_counts_no_outliers.columns = ['Sequence', 'N']

# Prepare the model formula using all unique sequences as predictors
sequences = '+'.join(f'Q("{seq}")' for seq in sequence_counts_no_outliers['Sequence'])

# Create a formula for OLS
formula = f'len_levels_complete ~ {sequences}'

# Convert the data to a format suitable for modeling
data_for_model = pd.get_dummies(data_no_outliers, columns=['Sequence'], prefix='', prefix_sep='')

# Fit the linear regression model using the formula API
model_no_outliers = ols(formula, data=data_for_model).fit()

# Create a summary table for coefficients, p-values, and confidence intervals
no_outliers_summary = model_no_outliers.summary2().tables[1]

# Merge the summary with the sequence counts
merged_summary = no_outliers_summary.reset_index()
merged_summary['Sequence'] = merged_summary['index'].apply(lambda x: x.split('[')[1][:-1] if 'Sequence' in x else 'const')
merged_summary = merged_summary.merge(sequence_counts_no_outliers, on='Sequence', how='left')

# Display the merged summary with coefficients and number of observations
merged_summary

/var/folders/t1/sdbl9_cd061431ynt1cns5hh0000gn/T/ipykernel_5158/626417676.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_no_outliers['Sequence'] = data_no_outliers.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)


,index,Coef.,Std.Err.,t,P>|t|,[0.025,0.975],Sequence,N
0,Intercept,32.632284,1.827043,17.860705,3.949567e-26,28.980076,36.284492,const,NaN
1,"Q(""low-returns_fast-reframe->low-returns_fast-...",7.948361,2.557523,3.107835,2.843757e-03,2.835945,13.060777,const,NaN
2,"Q(""low-returns_fast-reframe->low-returns_slow-...",2.185898,3.516286,0.621650,5.364515e-01,-4.843058,9.214853,const,NaN
3,"Q(""low-returns_fast-reframe->low-returns_fast-...",-2.298951,4.459381,-0.515531,6.080153e-01,-11.213127,6.615226,const,NaN
4,"Q(""low-returns_fast-reframe->high-returns_fast...",-1.032284,4.816190,-0.214336,8.309882e-01,-10.659710,8.595142,const,NaN
5,"Q(""low-returns_fast-reframe->low-returns_slow-...",5.117716,5.306607,0.964405,3.385898e-01,-5.490041,15.725473,const,NaN
6,"Q(""high-returns_fast-reframe->low-returns_fast...",2.867716,5.306607,0.540405,5.908540e-01,-7.740041,13.475473,const,NaN
7,"Q(""low-returns_fast-reframe->low-returns_fast-...",-0.632284,5.306607,-0.119150,9.055415e-01,-11.240041,9.975473,const,NaN
8,"Q(""low-returns_slow-reframe->low-returns_fast-...",-0.965617,6.036065,-0.159975,8.734211e-01,-13.031539,11.100304,const,NaN
9,"Q(""low-returns_fast-reframe->low-returns_slow-...",9.367716,7.278879,1.286972,2.028881e-01,-5.182555,23.917987,const,NaN


In [8]:
no_outliers_summary

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
Intercept,32.632284,1.827043,17.860705,3.949567e-26,28.980076,36.284492
"Q(""low-returns_fast-reframe->low-returns_fast-reframe->low-returns_fast-reframe"")[T.True]",7.948361,2.557523,3.107835,2.843757e-03,2.835945,13.060777
"Q(""low-returns_fast-reframe->low-returns_slow-reframe->low-returns_fast-reframe"")[T.True]",2.185898,3.516286,0.621650,5.364515e-01,-4.843058,9.214853
"Q(""low-returns_fast-reframe->low-returns_fast-reframe->high-returns_fast-reframe"")[T.True]",-2.298951,4.459381,-0.515531,6.080153e-01,-11.213127,6.615226
"Q(""low-returns_fast-reframe->high-returns_fast-reframe->low-returns_fast-reframe"")[T.True]",-1.032284,4.816190,-0.214336,8.309882e-01,-10.659710,8.595142
"Q(""low-returns_fast-reframe->low-returns_slow-reframe->high-returns_fast-reframe"")[T.True]",5.117716,5.306607,0.964405,3.385898e-01,-5.490041,15.725473
"Q(""high-returns_fast-reframe->low-returns_fast-reframe->low-returns_fast-reframe"")[T.True]",2.867716,5.306607,0.540405,5.908540e-01,-7.740041,13.475473
"Q(""low-returns_fast-reframe->low-returns_fast-reframe->low-returns_slow-reframe"")[T.True]",-0.632284,5.306607,-0.119150,9.055415e-01,-11.240041,9.975473
"Q(""low-returns_slow-reframe->low-returns_fast-reframe->low-returns_fast-reframe"")[T.True]",-0.965617,6.036065,-0.159975,8.734211e-01,-13.031539,11.100304
"Q(""low-returns_fast-reframe->low-returns_slow-reframe->low-returns_slow-reframe"")[T.True]",9.367716,7.278879,1.286972,2.028881e-01,-5.182555,23.917987


In [9]:
import pandas as pd
import statsmodels.api as sm

# Load the dataset
file_path = '/Users/baselhussein/Projects/impossible_goals/agg/data/merged_approaches.csv'
df = pd.read_csv(file_path)

# Remove outliers (3 SD from mean)
mean_levels_complete = df['len_levels_complete'].mean()
std_levels_complete = df['len_levels_complete'].std()
lower_threshold = mean_levels_complete - 3 * std_levels_complete
upper_threshold = mean_levels_complete + 3 * std_levels_complete

data_no_outliers = df[(df['len_levels_complete'] >= lower_threshold) & (df['len_levels_complete'] <= upper_threshold)]

# Create a 'Sequence' column to represent unique sequences
data_no_outliers['Sequence'] = data_no_outliers.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)

# Use pivot_table to create binary indicators for each unique sequence
sequence_indicators = pd.pivot_table(data_no_outliers, index=data_no_outliers.index, 
                                     columns='Sequence', aggfunc=len, fill_value=0)

# Combine the binary indicators with the original DataFrame
data_with_indicators = pd.concat([data_no_outliers[['len_levels_complete']], sequence_indicators], axis=1)

# Prepare predictors (X) and response variable (y)
X_no_outliers = data_with_indicators.drop(columns='len_levels_complete')
y_no_outliers = data_with_indicators['len_levels_complete']

# Add a constant term to the model for the intercept
X_no_outliers_with_intercept = sm.add_constant(X_no_outliers)

# Fit the linear regression model using statsmodels without outliers
model_no_outliers = sm.OLS(y_no_outliers, X_no_outliers_with_intercept)
results_no_outliers = model_no_outliers.fit()

# Extract coefficients, p-values, and confidence intervals for the model without outliers
no_outliers_summary = results_no_outliers.summary2().tables[1]

# Calculate the number of observations (N) for each unique sequence without outliers
sequence_counts_no_outliers = data_no_outliers['Sequence'].value_counts().reset_index()
sequence_counts_no_outliers.columns = ['Sequence', 'N']

# Merge the sequence counts with the regression summary
merged_summary = no_outliers_summary.reset_index().rename(columns={'index': 'Coefficient'})
merged_summary['Sequence'] = merged_summary['Coefficient'].apply(lambda x: x.split('[')[1][:-1] if 'Sequence' in x else 'const')
merged_summary = merged_summary.merge(sequence_counts_no_outliers, on='Sequence', how='left')

# Display the merged summary with coefficients and number of observations
merged_summary

/var/folders/t1/sdbl9_cd061431ynt1cns5hh0000gn/T/ipykernel_5158/3070750266.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_no_outliers['Sequence'] = data_no_outliers.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)


,Coefficient,Coef.,Std.Err.,t,P>|t|,[0.025,0.975],Sequence,N
0,const,26.416611,1.479035,17.860705,3.949567e-26,23.460062,29.373160,const,NaN
1,"(ID, high-returns_fast-reframe->high-returns_f...",0.516678,2.039580,0.253326,8.008547e-01,-3.560384,4.593739,const,NaN
2,"(ID, high-returns_fast-reframe->low-returns_fa...",0.916678,2.039580,0.449445,6.546783e-01,-3.160384,4.993739,const,NaN
3,"(ID, high-returns_fast-reframe->low-returns_fa...",1.816678,1.051474,1.727744,8.901261e-02,-0.285189,3.918544,const,NaN
4,"(ID, low-returns_fast-reframe->high-returns_fa...",2.716678,2.039580,1.331979,1.877428e-01,-1.360384,6.793739,const,NaN
...,...,...,...,...,...,...,...,...,...
76,"(lvl3, low-returns_slow-reframe->low-returns_f...",3.716678,2.039580,1.822277,7.323649e-02,-0.360384,7.793739,const,NaN
77,"(lvl3, low-returns_slow-reframe->low-returns_f...",1.050011,1.202066,0.873505,3.857584e-01,-1.352885,3.452907,const,NaN
78,"(lvl3, low-returns_slow-reframe->low-returns_f...",-0.283322,2.039580,-0.138912,8.899699e-01,-4.360384,3.793739,const,NaN
79,"(lvl3, low-returns_slow-reframe->low-returns_s...",1.116678,2.039580,0.547504,5.859982e-01,-2.960384,5.193739,const,NaN
